# Metrics Readiness Audit

This notebook summarizes the latest evaluation metrics and checks which ones meet a 0.95 threshold. It reads existing outputs under `reports/` and YOLO training runs, and does not train models.


In [ ]:
from pathlib import Path
import pandas as pd

threshold = 0.95
reports_dir = Path('reports')


In [ ]:
def _read_report_metrics(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['metric', 'value'])
    df = pd.read_csv(path)
    if 'Metric' in df.columns:
        metric_col = 'Metric'
    elif 'metric' in df.columns:
        metric_col = 'metric'
    else:
        metric_col = df.columns[0]
    value_col = 'mean' if 'mean' in df.columns else df.columns[1]
    out = df[[metric_col, value_col]].rename(columns={metric_col: 'metric', value_col: 'value'})
    out['value'] = pd.to_numeric(out['value'], errors='coerce')
    return out

def _best_yolo_metrics(paths):
    best = {}
    for p in paths:
        if not p.exists():
            continue
        df = pd.read_csv(p)
        for col in ('metrics/mAP50(B)', 'metrics/mAP50-95(B)'):
            if col not in df.columns:
                continue
            val = float(df[col].max())
            if col not in best or val > best[col][0]:
                best[col] = (val, str(p))
    return best


In [ ]:
report_files = {
    'fraud': reports_dir / 'metrics_fraud.csv',
    'cyber': reports_dir / 'metrics_cyber.csv',
    'behavior': reports_dir / 'metrics_behavior.csv',
    'vision': reports_dir / 'metrics_vision.csv',
    'fusion': reports_dir / 'metrics_fusion.csv',
    'voice': reports_dir / 'metrics_voice.csv',
    'video': reports_dir / 'metrics_video.csv',
    'recommender': reports_dir / 'metrics_recommender.csv',
}

metrics_of_interest = {
    'fraud': ['test_roc_auc', 'hybrid_roc_auc', 'test_f1', 'hybrid_f1'],
    'cyber': ['test_roc_auc', 'test_f1', 'anomaly_roc_auc', 'anomaly_f1'],
    'behavior': [
        'autoencoder_accuracy', 'autoencoder_f1', 'autoencoder_precision', 'autoencoder_recall',
        'autoencoder_pr_auc', 'autoencoder_roc_auc',
        'lof_accuracy', 'lof_f1', 'lof_precision', 'lof_recall', 'lof_pr_auc', 'lof_roc_auc',
    ],
    'vision': ['accuracy', 'f1', 'pr_auc', 'roc_auc'],
    'fusion': ['roc_auc', 'cv_roc_auc_mean', 'f1'],
    'voice': ['accuracy', 'f1', 'roc_auc'],
    'video': ['accuracy', 'f1', 'roc_auc'],
    'recommender': ['ndcg', 'recall', 'map', 'precision'],
}


In [ ]:
rows = []
for module, path in report_files.items():
    df = _read_report_metrics(path)
    if df.empty:
        for metric in metrics_of_interest.get(module, []):
            rows.append({'module': module, 'metric': metric, 'value': None, 'pass_95': False, 'source': str(path)})
        continue
    metric_set = set(metrics_of_interest.get(module, []))
    if not metric_set:
        metric_set = set(df['metric'].tolist())
    for metric in sorted(metric_set):
        match = df[df['metric'] == metric]
        value = match['value'].iloc[0] if not match.empty else None
        rows.append({'module': module, 'metric': metric, 'value': value, 'pass_95': value is not None and value >= threshold, 'source': str(path)})

# YOLO metrics
yolo_paths = [
    Path('models/brand/full/results.csv'),
    Path('models/brand/fast_run_1epoch/results.csv'),
    Path('runs/detect/train8/results.csv'),
    Path('runs/detect/train9/results.csv'),
    Path('runs/detect/train13/results.csv'),
]
best_yolo = _best_yolo_metrics(yolo_paths)
if best_yolo:
    for metric, (value, src) in best_yolo.items():
        rows.append({'module': 'brand_yolo', 'metric': metric, 'value': value, 'pass_95': value >= threshold, 'source': src})
else:
    rows.append({'module': 'brand_yolo', 'metric': 'metrics/mAP50-95(B)', 'value': None, 'pass_95': False, 'source': 'runs/detect'})

audit_df = pd.DataFrame(rows)
audit_df


In [ ]:
# Summary table per module (best metric among the listed ones)
summary_rows = []
for module in sorted(audit_df['module'].unique()):
    subset = audit_df[audit_df['module'] == module]
    subset = subset.dropna(subset=['value'])
    if subset.empty:
        summary_rows.append({'module': module, 'metric': 'missing', 'value': None, 'pass_95': False})
        continue
    best = subset.sort_values('value', ascending=False).iloc[0]
    summary_rows.append({
        'module': module,
        'metric': best['metric'],
        'value': float(best['value']),
        'pass_95': bool(best['pass_95']),
    })
summary = pd.DataFrame(summary_rows)
summary
